# Model Inference — Walmart Store Sales Forecasting

Loads the best registered model (**Light_GBM_Training_Best**) directly from the
MLflow Model Registry on Dagshub, rebuilds the exact feature pipeline used in
training, predicts on the raw Kaggle **test set**, and generates `submission.csv`.

### Why the feature pipeline is rebuilt here
The registered artifact is the bare LightGBM regressor (logged via
`mlflow.lightgbm.log_model`). The preprocessing (TimeSeriesFeatureBuilder →
feature transformers → IV/MI/correlation selection) therefore has to be applied
to the raw test data before prediction. The pipeline and selector are refit on
the same training data with the same thresholds as in
`model_experiment_Light_GBM.ipynb`, which deterministically reproduces the same
feature set in the same column order the model was trained on (the model was
fit on `.values`, so column order must match exactly).

### How lag features work for future test dates
The test set spans ~39 weeks after the last training date. Lag/rolling features
are computed over the concatenated train+test timeline: lags that reach back
into observed history get real values; deeper-future lags are NaN, which
LightGBM handles natively.


In [ ]:
!pip install kaggle==1.5.16

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! chmod 600 /root/.kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip -o Walmart-Recruiting-Store-Sales-Forecasting.zip
! unzip -o train.csv.zip
! unzip -o test.csv.zip
! unzip -o features.csv.zip
! unzip -o sampleSubmission.csv.zip

In [ ]:
!pip install -q dagshub mlflow lightgbm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
import dagshub
import mlflow
import mlflow.lightgbm
from mlflow.tracking import MlflowClient

# Same config as the training notebook — must match exactly so the refit
# pipeline reproduces the identical feature set and column order
TRAIN_PATH    = 'train.csv'
TEST_PATH     = 'test.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

LAGS            = [1, 2, 3, 4, 13, 26, 52]
ROLLING_WINDOWS = [4, 12, 26, 52]

VAL_START = '2012-04-01'

IV_THRESHOLD   = 0.02
MI_THRESHOLD   = 0.05
CORR_THRESHOLD = 0.90

REGISTERED_MODEL_NAME = 'Light_GBM_Training_Best'

print('Setup OK')

## Pipeline Class Definitions

Copied verbatim from `model_experiment_Light_GBM.ipynb` — the pipeline classes
must be byte-identical to training so the refit pipeline reproduces the exact
same feature set and column order the registered model was trained on.

In [ ]:
SUPERBOWL_DATES = pd.to_datetime([
    "2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"
])
LABORDAY_DATES = pd.to_datetime([
    "2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"
])
THANKSGIVING_DATES = pd.to_datetime([
    "2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"
])
CHRISTMAS_DATES = pd.to_datetime([
    "2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"
])

# Additional important dates
BACK_TO_SCHOOL_DATES = pd.to_datetime([
    "2010-08-27", "2011-08-26", "2012-08-31"
])
EASTER_DATES = pd.to_datetime([
    "2010-04-02", "2011-04-22", "2012-04-06"
])

In [ ]:
class TimeSeriesFeatureBuilder:

    def __init__(
        self,
        lags: list            = [1, 2, 3, 4, 13, 26, 52],   # added: 2, 3 (near-term), 13 (quarter), 26 (half-year)
        rolling_windows: list = [4, 12, 26, 52],
        group_cols: list      = ["Store", "Dept"],
        target_col: str       = "Weekly_Sales",
    ):
        self.lags            = lags
        self.rolling_windows = rolling_windows
        self.group_cols      = group_cols
        self.target_col      = target_col

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Run on the FULL merged dataframe before any train/val split."""
        df = df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df = df.sort_values(self.group_cols + ["Date"]).reset_index(drop=True)

        grp = df.groupby(self.group_cols)[self.target_col]

        # Lag features: what were sales N weeks ago?
        for lag in self.lags:
            df[f"sales_lag_{lag}"] = grp.shift(lag)

        # Rolling mean + std — shift(1) prevents leaking the current week
        shifted = grp.shift(1)
        for w in self.rolling_windows:
            df[f"rolling_mean_{w}"] = shifted.transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            )
            df[f"rolling_std_{w}"] = shifted.transform(
                lambda x: x.rolling(w, min_periods=1).std()
            )
            # min/max: range of sales over the window
            df[f"rolling_min_{w}"] = shifted.transform(
                lambda x: x.rolling(w, min_periods=1).min()
            )
            df[f"rolling_max_{w}"] = shifted.transform(
                lambda x: x.rolling(w, min_periods=1).max()
            )
            # median: robust to outlier holiday spikes
            df[f"rolling_median_{w}"] = shifted.transform(
                lambda x: x.rolling(w, min_periods=1).median()
            )
            # skew: detects asymmetric distributions (holiday spikes push right)
            df[f"rolling_skew_{w}"] = shifted.transform(
                lambda x: x.rolling(w, min_periods=2).skew()
            )

        # Coefficient of Variation — relative volatility, scale-independent
        df["rolling_cv_4"] = (
            df["rolling_std_4"] / df["rolling_mean_4"].replace(0, np.nan)
        )

        # Momentum — short-term vs long-term trend ratio: >1 means accelerating
        df["sales_momentum"] = (
            df["rolling_mean_4"] / df["rolling_mean_52"].replace(0, np.nan)
        )

        # EWMA: weights recent weeks more than plain rolling mean
        df["ewma_4"]  = shifted.transform(lambda x: x.ewm(span=4,  min_periods=1).mean())
        df["ewma_12"] = shifted.transform(lambda x: x.ewm(span=12, min_periods=1).mean())

        # YoY: removes long-term trend, improves stationarity
        if 52 in self.lags:
            df["yoy_diff"]        = df[self.target_col] - df["sales_lag_52"]
            df["yoy_growth_rate"] = (
                df[self.target_col] / df["sales_lag_52"].replace(0, np.nan) - 1
            )

        # Days since last markdown event (per store, not per dept)
        md_cols = [c for c in
                   ["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]
                   if c in df.columns]
        if md_cols:
            df["_any_md"] = df[md_cols].notna().any(axis=1)
            df["days_since_markdown"] = (
                df.groupby("Store")["_any_md"]
                .transform(lambda x:
                    x.cumsum().where(x).ffill().rsub(x.cumsum()).mul(7)
                )
            )
            df.drop(columns=["_any_md"], inplace=True)

        # External temporal features — require sorted history, so computed here
        if "Fuel_Price" in df.columns:
            fp_prev = df.groupby("Store")["Fuel_Price"].shift(1)
            df["fuel_price_chg"]     = df["Fuel_Price"] - fp_prev
            df["fuel_price_chg_pct"] = df["fuel_price_chg"] / fp_prev.replace(0, np.nan)

        if "CPI" in df.columns:
            cpi_52 = df.groupby("Store")["CPI"].shift(52)
            df["cpi_yoy"] = df["CPI"] / cpi_52.replace(0, np.nan) - 1

        if "Unemployment" in df.columns:
            df["unemp_delta"] = (
                df["Unemployment"] - df.groupby("Store")["Unemployment"].shift(4)
            )

        self._fitted = True
        return df

    def transform_incremental(
        self, df_history: pd.DataFrame, df_new: pd.DataFrame
    ) -> pd.DataFrame:
        """
        Inference only: appends new week(s) to history, computes time
        features, returns only the new rows.
        """
        if not hasattr(self, "_fitted"):
            raise RuntimeError("Call fit_transform on full data first.")
        time_prefixes = [
            "sales_lag", "rolling_", "ewma_", "yoy_", "days_since",
            "fuel_price_", "cpi_yoy", "unemp_delta", "sales_momentum",
        ]
        time_cols = [c for c in df_history.columns
                     if any(c.startswith(p) for p in time_prefixes)]
        combined    = pd.concat(
            [df_history.drop(columns=time_cols, errors="ignore"), df_new],
            ignore_index=True,
        )
        combined_fe = self.fit_transform(combined)
        return combined_fe.iloc[-len(df_new):].reset_index(drop=True)

In [ ]:
class CalendarFeatureTransformer(BaseEstimator, TransformerMixin):
    """
    Extracts week, month, quarter from Date.
    Adds sin/cos cyclical encodings so week 52 → week 1 is a small step.
    Adds binary flags for each specific holiday and proximity features
    (weeks_to_thanksgiving, pre_thanksgiving, pre_christmas, is_january).
    Added: is_payweek, is_black_friday, is_post_thanksgiving,
           is_post_christmas, is_tax_refund_season, is_summer.
    """

    def fit(self, X, y=None):
        self.fitted_ = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X["Date"] = pd.to_datetime(X["Date"])

        X["week"]          = X["Date"].dt.isocalendar().week.astype(int)
        X["month"]         = X["Date"].dt.month
        X["year"]          = X["Date"].dt.year
        X["quarter"]       = X["Date"].dt.quarter
        X["week_of_month"] = (X["Date"].dt.day - 1) // 7 + 1

        # Cyclical: prevents the model treating week 52 → week 1 as a big jump
        X["week_sin"]  = np.sin(2 * np.pi * X["week"]  / 52)
        X["week_cos"]  = np.cos(2 * np.pi * X["week"]  / 52)
        X["month_sin"] = np.sin(2 * np.pi * X["month"] / 12)
        X["month_cos"] = np.cos(2 * np.pi * X["month"] / 12)

        # Which specific holiday is this week?
        X["is_superbowl"]      = X["Date"].isin(SUPERBOWL_DATES).astype(int)
        X["is_laborday"]       = X["Date"].isin(LABORDAY_DATES).astype(int)
        X["is_thanksgiving"]   = X["Date"].isin(THANKSGIVING_DATES).astype(int)
        X["is_christmas"]      = X["Date"].isin(CHRISTMAS_DATES).astype(int)
        X["is_back_to_school"] = X["Date"].isin(BACK_TO_SCHOOL_DATES).astype(int)
        X["is_easter"]         = X["Date"].isin(EASTER_DATES).astype(int)

        def weeks_to_nearest(dates_series, holiday_dates):
            return [
                min([(d - h).days // 7 for h in holiday_dates], key=abs)
                for d in dates_series
            ]

        X["weeks_to_thanksgiving"] = np.clip(
            weeks_to_nearest(X["Date"], THANKSGIVING_DATES), -8, 8
        )
        X["weeks_to_christmas"] = np.clip(
            weeks_to_nearest(X["Date"], CHRISTMAS_DATES), -8, 8
        )
        X["pre_thanksgiving"] = X["weeks_to_thanksgiving"].between(-4, -1).astype(int)
        X["pre_christmas"]    = X["weeks_to_christmas"].between(-4, -1).astype(int)
        X["is_january"]       = (X["month"] == 1).astype(int)

        # Pay period: customers tend to shop when they receive their paycheck
        X["is_payweek"] = (
            (X["Date"].dt.day <= 7) | X["Date"].dt.day.between(14, 21)
        ).astype(int)

        # Black Friday: the week immediately after Thanksgiving — peak sales
        X["is_black_friday"] = (X["weeks_to_thanksgiving"] == 1).astype(int)

        # Post-Thanksgiving hangover: 1-2 weeks after, demand drops back
        X["is_post_thanksgiving"] = X["weeks_to_thanksgiving"].between(1, 2).astype(int)

        # Post-Christmas: early January sees returns and markdown clearance
        X["is_post_christmas"] = (
            (X["month"] == 1) & (X["Date"].dt.day <= 14)
        ).astype(int)

        # Tax refund season: Feb-Mar IRS refunds give consumers extra spending power
        X["is_tax_refund_season"] = X["month"].isin([2, 3]).astype(int)

        # Summer months: Jun-Aug — seasonal categories shift (outdoor, back-to-school prep)
        X["is_summer"] = X["month"].isin([6, 7, 8]).astype(int)

        return X

In [ ]:
class MarkdownFeatureTransformer(BaseEstimator, TransformerMixin):
    """
    MarkDown1..5 are ~66% null because promotions don't run every week.
    Aggregates them into summary columns:
      total_markdown     — total promotional spend this week
      markdown_active    — binary: was any promotion running?
      markdown_count     — how many of the 5 types were active?
      max_single_md      — size of the largest single discount
      total_markdown_log — log-scaled total (corrects right skew)
      markdown_diversity — fraction of types active (count / 5)
      md{i}_active       — per-type binary flags (each type has different mechanism)
    """

    def __init__(self, drop_raw: bool = True):
        self.drop_raw = drop_raw

    def fit(self, X, y=None):
        self.fitted_ = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X   = X.copy()
        mdc = [c for c in ["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]
               if c in X.columns]
        if not mdc:
            return X
        md = X[mdc].fillna(0)

        X["total_markdown"]     = md.sum(axis=1)
        X["markdown_active"]    = (X["total_markdown"] > 0).astype(int)
        X["markdown_count"]     = (md > 0).sum(axis=1)
        X["max_single_md"]      = md.max(axis=1)
        X["total_markdown_log"] = np.log1p(X["total_markdown"].values.astype(float))

        # Fraction of markdown types that are active: 0.0 (none) – 1.0 (all five)
        X["markdown_diversity"] = X["markdown_count"] / len(mdc)

        # Per-type presence flags — each MarkDown type covers a different promotion channel
        for col in mdc:
            flag_name = col.lower().replace("markdown", "md") + "_active"
            X[flag_name] = (X[col].fillna(0) > 0).astype(int)

        if self.drop_raw:
            X.drop(columns=mdc, inplace=True)
        return X

In [ ]:
class StoreFeatureTransformer(BaseEstimator, TransformerMixin):
    """
    Derives features from Type (A/B/C) and Size (sq footage).
    Computes dept_sales_share: each department's historical fraction
    of its store's total sales — fitted on train only, no leakage.
    Unseen (Store, Dept) pairs at inference fall back to training mean.
    Added: dept_volatility_cv, dept_rank_in_store, store_holiday_sensitivity.
    """

    def fit(self, X: pd.DataFrame, y=None):
        self.fitted_ = True
        if "Weekly_Sales" in X.columns:
            store_avg        = X.groupby("Store")["Weekly_Sales"].mean()
            dept_avg         = X.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
            self.dept_share_ = (dept_avg / store_avg).rename("dept_sales_share")

            # Dept volatility: CV = std/mean — how erratic is this dept vs its own average?
            dept_std = X.groupby(["Store", "Dept"])["Weekly_Sales"].std()
            self.dept_volatility_ = (
                dept_std / dept_avg.replace(0, np.nan)
            ).rename("dept_volatility_cv")

            # Dept rank within its store by average sales (1 = top revenue dept)
            self.dept_rank_ = (
                dept_avg
                .groupby(level="Store")
                .rank(ascending=False, method="min")
                .rename("dept_rank_in_store")
            )

            # Store-level holiday sensitivity: ratio of mean holiday vs non-holiday sales
            if "IsHoliday" in X.columns:
                hol_avg  = X[X["IsHoliday"] == 1].groupby("Store")["Weekly_Sales"].mean()
                nhol_avg = X[X["IsHoliday"] == 0].groupby("Store")["Weekly_Sales"].mean()
                self.store_holiday_sensitivity_ = (
                    hol_avg / nhol_avg.replace(0, np.nan)
                ).rename("store_holiday_sensitivity")
            else:
                self.store_holiday_sensitivity_ = None
        else:
            self.dept_share_                = None
            self.dept_volatility_           = None
            self.dept_rank_                 = None
            self.store_holiday_sensitivity_ = None
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        if "Size" in X.columns:
            X["store_size_log"] = np.log1p(X["Size"])
        if "Type" in X.columns:
            X["store_type_ord"] = (
                X["Type"].map({"A": 2, "B": 1, "C": 0}).fillna(-1).astype(int)
            )
        if self.dept_share_ is not None:
            X = X.join(self.dept_share_, on=["Store", "Dept"], how="left")
            X["dept_sales_share"] = X["dept_sales_share"].fillna(self.dept_share_.mean())

        if self.dept_volatility_ is not None:
            X = X.join(self.dept_volatility_, on=["Store", "Dept"], how="left")
            X["dept_volatility_cv"] = X["dept_volatility_cv"].fillna(self.dept_volatility_.mean())

        if self.dept_rank_ is not None:
            X = X.join(self.dept_rank_, on=["Store", "Dept"], how="left")
            X["dept_rank_in_store"] = X["dept_rank_in_store"].fillna(self.dept_rank_.max())

        if self.store_holiday_sensitivity_ is not None:
            X = X.join(self.store_holiday_sensitivity_, on="Store", how="left")
            X["store_holiday_sensitivity"] = X["store_holiday_sensitivity"].fillna(
                self.store_holiday_sensitivity_.mean()
            )

        return X

In [ ]:
class ExternalFeatureTransformer(BaseEstimator, TransformerMixin):
    """
    Handles Temperature, Fuel_Price, CPI, Unemployment.
    Fills rare nulls with training medians (fitted, no leakage).
    Adds temp_bucket: 0=cold / 1=mild / 2=warm / 3=hot.
    Added: is_extreme_cold (<=20°F), is_extreme_heat (>=90°F).
    Note: temporal external features (fuel_price_chg, cpi_yoy, unemp_delta)
          are computed in TimeSeriesFeatureBuilder since they require shift().
    """

    def fit(self, X: pd.DataFrame, y=None):
        self.fitted_  = True
        self.ext_cols_ = [c for c in ["Temperature","Fuel_Price","CPI","Unemployment"]
                          if c in X.columns]
        self.medians_  = X[self.ext_cols_].median()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        for col in self.ext_cols_:
            X[col] = X[col].fillna(self.medians_[col])
        if "Temperature" in X.columns:
            X["temp_bucket"] = pd.cut(
                X["Temperature"],
                bins=[-999, 32, 60, 80, 999],
                labels=[0, 1, 2, 3],
            ).astype(float)
            # Extreme weather reduces foot traffic regardless of season bucket
            X["is_extreme_cold"] = (X["Temperature"] <= 20).astype(int)
            X["is_extreme_heat"] = (X["Temperature"] >= 90).astype(int)
        return X

In [ ]:
class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    """
    Drops raw columns superseded by engineered ones,
    plus the target and the Date string.
    """

    def __init__(self, cols_to_drop: list = None):
        self.cols_to_drop = cols_to_drop or ["Date", "Type", "Size", "Weekly_Sales"]

    def fit(self, X, y=None):
        self.fitted_ = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X.drop(columns=[c for c in self.cols_to_drop if c in X.columns])


def build_feature_pipeline(drop_raw_markdowns: bool = True) -> Pipeline:
    """Assembles the five row-independent transformers into a single sklearn Pipeline."""
    return Pipeline(steps=[
        ("calendar",  CalendarFeatureTransformer()),
        ("markdown",  MarkdownFeatureTransformer(drop_raw=drop_raw_markdowns)),
        ("store",     StoreFeatureTransformer()),
        ("external",  ExternalFeatureTransformer()),
        ("drop_cols", DropColumnsTransformer()),
    ])

In [ ]:
def _compute_iv_for_feature(
    feature_values: np.ndarray,
    target_bins: np.ndarray,
    n_target_bins: int,
    n_feature_bins: int = 10,
    feature_is_binary: bool = False,
) -> tuple:
    """
    Bins a feature then computes WOE and IV against a pre-binned target.
    Returns (total_iv: float, detail_table: DataFrame).
    The 'event' is defined as landing in the top sales decile.
    """
    if feature_is_binary or len(np.unique(feature_values)) <= 2:
        feature_bins = feature_values.astype(int)
    else:
        kbd = KBinsDiscretizer(n_bins=n_feature_bins, encode="ordinal", strategy="quantile")
        try:
            feature_bins = kbd.fit_transform(
                feature_values.reshape(-1, 1)
            ).ravel().astype(int)
        except Exception:
            feature_bins = (
                pd.qcut(feature_values,
                        q=min(n_feature_bins, len(np.unique(feature_values))),
                        labels=False, duplicates="drop")
                .astype(float).fillna(0).astype(int).values
            )

    event_bin    = n_target_bins - 1
    is_event     = (target_bins == event_bin).astype(int)
    total_events = is_event.sum()
    total_non    = len(is_event) - total_events
    if total_events == 0 or total_non == 0:
        return 0.0, pd.DataFrame()

    rows = []
    for bucket in np.unique(feature_bins):
        mask  = feature_bins == bucket
        n_ev  = is_event[mask].sum()
        n_no  = (~is_event[mask].astype(bool)).sum()
        d_ev  = (n_ev + 0.5) / (total_events + 0.5)  # Laplace smoothing
        d_no  = (n_no + 0.5) / (total_non   + 0.5)
        woe   = np.log(d_ev / d_no)
        rows.append({"bucket": bucket, "n_event": n_ev, "n_non": n_no,
                     "woe": woe, "iv": (d_ev - d_no) * woe})

    table = pd.DataFrame(rows)
    return float(table["iv"].sum()), table

In [ ]:
class WOEIVFilter(BaseEstimator, TransformerMixin):
    """
    Bins the continuous target into quantile deciles, computes Information
    Value for every feature, drops features below iv_threshold.

    IV thresholds (standard credit-scoring convention):
      < 0.02   — useless
      0.02–0.1 — weak
      0.1–0.3  — medium
      > 0.3    — strong
    """

    def __init__(self, iv_threshold=0.02, n_target_bins=10, n_feature_bins=10):
        self.iv_threshold   = iv_threshold
        self.n_target_bins  = n_target_bins
        self.n_feature_bins = n_feature_bins

    def fit(self, X: pd.DataFrame, y: np.ndarray):
        self.fitted_           = True
        self.feature_names_in_ = list(X.columns)

        target_kbd  = KBinsDiscretizer(
            n_bins=self.n_target_bins, encode="ordinal", strategy="quantile"
        )
        target_bins = target_kbd.fit_transform(
            np.clip(np.array(y), 0, None).reshape(-1, 1)
        ).ravel().astype(int)

        iv_scores, self.iv_tables_ = {}, {}
        for col in X.columns:
            vals      = X[col].fillna(0).values.astype(float)
            is_binary = set(np.unique(vals)).issubset({0.0, 1.0})
            iv, tbl   = _compute_iv_for_feature(
                vals, target_bins,
                n_target_bins=self.n_target_bins,
                n_feature_bins=self.n_feature_bins,
                feature_is_binary=is_binary,
            )
            iv_scores[col]       = iv
            self.iv_tables_[col] = tbl

        self.iv_scores_         = pd.Series(iv_scores).sort_values(ascending=False)
        self.selected_features_ = list(self.iv_scores_[self.iv_scores_ >= self.iv_threshold].index)
        self.dropped_features_  = list(self.iv_scores_[self.iv_scores_ <  self.iv_threshold].index)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        missing = [c for c in self.selected_features_ if c not in X.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        return X[self.selected_features_].copy()

    @property
    def iv_summary(self) -> pd.DataFrame:
        df = self.iv_scores_.reset_index()
        df.columns = ["feature", "iv"]
        df["strength"] = pd.cut(df["iv"],
            bins=[-np.inf, 0.02, 0.1, 0.3, np.inf],
            labels=["useless", "weak", "medium", "strong"])
        df["kept"] = df["iv"] >= self.iv_threshold
        return df

In [ ]:
class MutualInfoFilter(BaseEstimator, TransformerMixin):
    """
    Computes mutual information between each feature and the continuous
    target — the proper regression-native alternative to IV.
    Drops features whose MI < mi_threshold * max(MI scores).

    mi_threshold=0.05 means: keep features that carry at least 5%
    of the information the strongest feature carries.
    """

    def __init__(self, mi_threshold=0.05, random_state=42):
        self.mi_threshold = mi_threshold
        self.random_state = random_state

    def fit(self, X: pd.DataFrame, y: np.ndarray):
        self.fitted_           = True
        self.feature_names_in_ = list(X.columns)

        mi_raw          = mutual_info_regression(
            X.fillna(0), y, random_state=self.random_state
        )
        self.mi_scores_ = pd.Series(mi_raw, index=X.columns).sort_values(ascending=False)

        cutoff                  = self.mi_scores_.max() * self.mi_threshold
        self.selected_features_ = list(self.mi_scores_[self.mi_scores_ >= cutoff].index)
        self.dropped_features_  = list(self.mi_scores_[self.mi_scores_ <  cutoff].index)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X[[c for c in self.selected_features_ if c in X.columns]].copy()

In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Removes one of any pair of features with |Pearson r| >= corr_threshold.
    When a pair conflicts, the feature with the lower importance score
    is dropped (importance comes from IV scores by default).
    """

    def __init__(self, corr_threshold=0.90, importance: pd.Series = None):
        self.corr_threshold = corr_threshold
        self.importance     = importance

    def fit(self, X: pd.DataFrame, y=None):
        self.fitted_           = True
        self.feature_names_in_ = list(X.columns)
        self.corr_matrix_      = X.fillna(0).corr().abs()

        importance = (
            self.importance.reindex(X.columns).fillna(0)
            if self.importance is not None
            else pd.Series(range(len(X.columns), 0, -1), index=X.columns, dtype=float)
        )

        to_drop = set()
        cols = list(X.columns)
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                ci, cj = cols[i], cols[j]
                if ci in to_drop or cj in to_drop:
                    continue
                if self.corr_matrix_.loc[ci, cj] >= self.corr_threshold:
                    to_drop.add(cj if importance.get(ci, 0) >= importance.get(cj, 0) else ci)

        self.dropped_features_  = sorted(to_drop)
        self.selected_features_ = [c for c in cols if c not in to_drop]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X[[c for c in self.selected_features_ if c in X.columns]].copy()

    def correlated_pairs(self) -> pd.DataFrame:
        rows = []
        cols = self.feature_names_in_
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                r = self.corr_matrix_.loc[cols[i], cols[j]]
                if r >= self.corr_threshold:
                    dropped = cols[i] if cols[i] in self.dropped_features_ else cols[j]
                    rows.append({"feature_a": cols[i], "feature_b": cols[j],
                                 "correlation": round(r, 4),
                                 "kept":    cols[j] if dropped == cols[i] else cols[i],
                                 "dropped": dropped})
        return pd.DataFrame(rows).sort_values("correlation", ascending=False)

In [ ]:
class FeatureSelector(BaseEstimator, TransformerMixin):
    """
    Chains WOEIVFilter → MutualInfoFilter → CorrelationFilter.
    Each filter receives only the features that survived the previous one.
    The correlation filter uses IV scores as its importance ranking.
    """

    def __init__(
        self,
        iv_threshold:   float = 0.02,
        mi_threshold:   float = 0.05,
        corr_threshold: float = 0.90,
        n_target_bins:  int   = 10,
        n_feature_bins: int   = 10,
        run_iv:   bool = True,
        run_mi:   bool = True,
        run_corr: bool = True,
    ):
        self.iv_threshold   = iv_threshold
        self.mi_threshold   = mi_threshold
        self.corr_threshold = corr_threshold
        self.n_target_bins  = n_target_bins
        self.n_feature_bins = n_feature_bins
        self.run_iv         = run_iv
        self.run_mi         = run_mi
        self.run_corr       = run_corr

    def fit(self, X: pd.DataFrame, y: np.ndarray):
        self.fitted_           = True
        self.initial_features_ = list(X.columns)
        X_curr = X.copy()
        y      = np.array(y)

        self.iv_filter_ = None
        if self.run_iv:
            self.iv_filter_ = WOEIVFilter(
                self.iv_threshold, self.n_target_bins, self.n_feature_bins
            )
            self.iv_filter_.fit(X_curr, y)
            X_curr = self.iv_filter_.transform(X_curr)
            print(f"[IV filter]   {len(self.initial_features_):3d} → {len(X_curr.columns):3d}  "
                  f"dropped: {self.iv_filter_.dropped_features_}")

        self.mi_filter_ = None
        if self.run_mi:
            self.mi_filter_ = MutualInfoFilter(self.mi_threshold)
            self.mi_filter_.fit(X_curr, y)
            X_curr = self.mi_filter_.transform(X_curr)
            print(f"[MI filter]       → {len(X_curr.columns):3d}  "
                  f"dropped: {self.mi_filter_.dropped_features_}")

        self.corr_filter_ = None
        if self.run_corr:
            importance = (
                self.iv_filter_.iv_scores_ if self.iv_filter_ is not None else
                self.mi_filter_.mi_scores_ if self.mi_filter_ is not None else None
            )
            self.corr_filter_ = CorrelationFilter(self.corr_threshold, importance)
            self.corr_filter_.fit(X_curr)
            X_curr = self.corr_filter_.transform(X_curr)
            print(f"[Corr filter]     → {len(X_curr.columns):3d}  "
                  f"dropped: {self.corr_filter_.dropped_features_}")

        self.selected_features_ = list(X_curr.columns)
        print(f"Result: {len(self.initial_features_)} → {len(self.selected_features_)} features kept")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X_curr = X.copy()
        if self.iv_filter_:   X_curr = self.iv_filter_.transform(X_curr)
        if self.mi_filter_:   X_curr = self.mi_filter_.transform(X_curr)
        if self.corr_filter_: X_curr = self.corr_filter_.transform(X_curr)
        return X_curr

    def summary(self) -> pd.DataFrame:
        rows = []
        for feat in self.initial_features_:
            iv = self.iv_filter_.iv_scores_.get(feat, np.nan) if self.iv_filter_ else np.nan
            mi = self.mi_filter_.mi_scores_.get(feat, np.nan) if self.mi_filter_ else np.nan
            if feat not in self.selected_features_:
                if self.iv_filter_     and feat in self.iv_filter_.dropped_features_:   by = "IV"
                elif self.mi_filter_   and feat in self.mi_filter_.dropped_features_:   by = "MI"
                elif self.corr_filter_ and feat in self.corr_filter_.dropped_features_: by = "Corr"
                else: by = "unknown"
            else:
                by = None
            rows.append({"feature": feat,
                         "iv": round(iv, 4) if not np.isnan(iv) else np.nan,
                         "mi": round(mi, 4) if not np.isnan(mi) else np.nan,
                         "kept": feat in self.selected_features_,
                         "dropped_by": by})
        return (pd.DataFrame(rows)
                .sort_values(["kept", "iv"], ascending=[False, False])
                .reset_index(drop=True))

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, weights: np.ndarray) -> float:
    """
    Weighted Mean Absolute Error — the Walmart competition metric.
    Holiday weeks carry weight=5, non-holiday weeks weight=1.
    """
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def make_holiday_weights(is_holiday: pd.Series) -> np.ndarray:
    """Convert IsHoliday boolean series to a WMAE weight array (5.0 or 1.0)."""
    return np.where(is_holiday.values, 5.0, 1.0)


def predict_new_week(
    df_history: pd.DataFrame,
    df_new_week: pd.DataFrame,
    builder: TimeSeriesFeatureBuilder,
    feature_pipeline: Pipeline,
    selector: FeatureSelector,
    model,
) -> np.ndarray:
    """
    End-to-end inference on a new unseen week.
    Uses the already-fitted builder, pipeline, and selector — no re-fitting.
    """
    df_fe = builder.transform_incremental(df_history, df_new_week)
    X_new = feature_pipeline.transform(df_fe)
    X_sel = selector.transform(X_new)
    return model.predict(X_sel.values)


def plot_iv(iv_filter: WOEIVFilter, threshold: float) -> None:
    """Horizontal bar chart of IV scores, colour-coded by strength tier."""
    iv_df = iv_filter.iv_summary.sort_values("iv", ascending=True)
    color_map = {
        "useless": "#d9534f", "weak": "#f0ad4e",
        "medium":  "#5bc0de", "strong": "#5cb85c",
    }
    colors = [color_map[str(s)] for s in iv_df["strength"]]

    fig, ax = plt.subplots(figsize=(8, max(4, len(iv_df) * 0.28)))
    ax.barh(iv_df["feature"], iv_df["iv"], color=colors, edgecolor="none", height=0.7)
    ax.axvline(threshold, color="red", lw=1.2, ls="--", label=f"threshold = {threshold}")

    from matplotlib.patches import Patch
    legend_els = [Patch(facecolor=v, label=k) for k, v in color_map.items()]
    legend_els.append(plt.Line2D([0],[0], color="red", ls="--",
                                 label=f"threshold = {threshold}"))
    ax.legend(handles=legend_els, fontsize=8, loc="lower right")
    ax.set_xlabel("Information Value (IV)")
    ax.set_title("Feature IV scores")
    plt.tight_layout()
    plt.show()


def plot_mi(mi_filter: MutualInfoFilter, threshold_frac: float) -> None:
    """Horizontal bar chart of MI scores."""
    mi_df  = mi_filter.mi_scores_.reset_index()
    mi_df.columns = ["feature", "mi"]
    mi_df  = mi_df.sort_values("mi", ascending=True)
    cutoff = mi_filter.mi_scores_.max() * threshold_frac
    colors = ["#5cb85c" if v >= cutoff else "#d9534f" for v in mi_df["mi"]]

    fig, ax = plt.subplots(figsize=(8, max(4, len(mi_df) * 0.28)))
    ax.barh(mi_df["feature"], mi_df["mi"], color=colors, edgecolor="none", height=0.7)
    ax.axvline(cutoff, color="red", lw=1.2, ls="--", label=f"cutoff = {cutoff:.4f}")
    ax.set_xlabel("Mutual Information score")
    ax.set_title("Feature MI scores  (green = kept)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_corr_heatmap(X: pd.DataFrame) -> None:
    """Correlation heatmap of the surviving features."""
    corr = X.corr()
    fig, ax = plt.subplots(figsize=(max(6, len(corr) * 0.55), max(5, len(corr) * 0.5)))
    sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)),
                annot=True, fmt=".2f", cmap="RdYlGn_r",
                vmin=-1, vmax=1, linewidths=0.4, ax=ax, annot_kws={"size": 7})
    ax.set_title(f"Correlation matrix — {len(corr)} surviving features")
    plt.tight_layout()
    plt.show()

## 1. Load Best Model from MLflow Model Registry

In [ ]:
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

client = MlflowClient()

# Load the latest registered version directly from the Model Registry,
# as required by the assignment
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
latest   = max(versions, key=lambda v: int(v.version))

print(f'Registered model : {REGISTERED_MODEL_NAME}')
print(f'Latest version   : {latest.version}')
print(f'Source run       : {latest.run_id}')
print(f'Tags             : {latest.tags}')

model = mlflow.lightgbm.load_model(f'models:/{REGISTERED_MODEL_NAME}/{latest.version}')
print(f'\nModel loaded: {type(model).__name__}')
print(f'Expects {model.n_features_} input features')

## 2. Rebuild the Feature Pipeline on Raw Data

Train and test are concatenated **before** the lag/rolling feature build so
test rows can look back into observed training history for their lag values.
The pipeline and selector are then fit **only on training rows** (same split
and thresholds as the training notebook) — test rows are only transformed.

In [ ]:
train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
test_raw     = pd.read_csv(TEST_PATH,     parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

print(f'Train : {train_raw.shape}  ({train_raw["Date"].min().date()} → {train_raw["Date"].max().date()})')
print(f'Test  : {test_raw.shape}  ({test_raw["Date"].min().date()} → {test_raw["Date"].max().date()})')

# Concatenate train + test (test has no Weekly_Sales → NaN target)
test_raw['Weekly_Sales'] = np.nan
test_raw['is_test']      = True
train_raw['is_test']     = False
combined = pd.concat([train_raw, test_raw], ignore_index=True)

# Merge features + stores exactly as in training
df_merged = (
    combined
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
    .sort_values(['Store', 'Dept', 'Date'])
    .reset_index(drop=True)
)

print(f'\nCombined merged: {df_merged.shape}')

In [ ]:
# Stage 1: lag / rolling features over the combined timeline
# Test rows within 52 weeks of the train end get real lag values;
# deeper-future lags remain NaN (LightGBM handles NaN natively).
builder = TimeSeriesFeatureBuilder(lags=LAGS, rolling_windows=ROLLING_WINDOWS)
df_feat = builder.fit_transform(df_merged)

new_cols = [c for c in df_feat.columns if c not in df_merged.columns]
print(f'Time-series features added: {len(new_cols)}')

# Split back into train / test portions
df_train_part = df_feat[~df_feat['is_test']].copy()
df_test_part  = df_feat[df_feat['is_test']].copy()

print(f'Train portion : {df_train_part.shape}')
print(f'Test portion  : {df_test_part.shape}')

In [ ]:
# Stage 2: fit the row-transform pipeline on TRAIN rows only, transform both
feature_pipeline = build_feature_pipeline()
feature_pipeline.fit(df_train_part)

X_train_df = feature_pipeline.transform(df_train_part)
X_test_df  = feature_pipeline.transform(df_test_part)

y_train = df_train_part['Weekly_Sales'].values

print(f'After row transforms — train: {X_train_df.shape}, test: {X_test_df.shape}')

# Stage 3: refit feature selection on TRAIN rows only (same thresholds
# as training notebook → deterministically reproduces the same columns)
selector = FeatureSelector(
    iv_threshold   = IV_THRESHOLD,
    mi_threshold   = MI_THRESHOLD,
    corr_threshold = CORR_THRESHOLD,
)
selector.fit(X_train_df, y_train)

X_train_final = selector.transform(X_train_df)
X_test_final  = selector.transform(X_test_df)

print(f'After feature selection — train: {X_train_final.shape}, test: {X_test_final.shape}')

# ── Sanity check: the model must see exactly the features it was trained on ──
assert X_test_final.shape[1] == model.n_features_, (
    f'Feature count mismatch: pipeline produced {X_test_final.shape[1]} features '
    f'but the registered model expects {model.n_features_}. '
    f'Check that LAGS / ROLLING_WINDOWS / thresholds match the training notebook exactly.'
)
print(f'\n✓ Feature count matches model expectation ({model.n_features_})')

## 3. Predict on the Test Set

In [ ]:
predictions = model.predict(X_test_final.values)

# Negative sales predictions are clipped to 0 — Weekly_Sales in the training
# data can technically be slightly negative (returns), but large negatives
# from NaN-heavy far-future rows are artifacts, and clipping at 0 is the
# standard, slightly-safer choice for this competition
predictions = np.clip(predictions, 0, None)

df_test_part = df_test_part.reset_index(drop=True)
df_test_part['Weekly_Sales_pred'] = predictions

print(f'Predictions generated: {len(predictions):,}')
print(f'Prediction stats: min={predictions.min():,.2f}  '
      f'mean={predictions.mean():,.2f}  max={predictions.max():,.2f}')

## 4. Generate Kaggle Submission

The submission format requires `Id = Store_Dept_Date` and `Weekly_Sales`.

In [ ]:
submission = pd.DataFrame({
    'Id': (
        df_test_part['Store'].astype(str) + '_' +
        df_test_part['Dept'].astype(str)  + '_' +
        df_test_part['Date'].dt.strftime('%Y-%m-%d')
    ),
    'Weekly_Sales': df_test_part['Weekly_Sales_pred'],
})

# Verify against the official sample submission: same Ids, same order
sample = pd.read_csv('sampleSubmission.csv')
print(f'Sample submission rows : {len(sample):,}')
print(f'Our submission rows    : {len(submission):,}')

# Align to the sample's Id order (Kaggle requires exact Id coverage)
submission = (
    sample[['Id']]
    .merge(submission, on='Id', how='left')
)

missing = submission['Weekly_Sales'].isna().sum()
if missing > 0:
    print(f'WARNING: {missing} Ids had no prediction — filling with 0')
    submission['Weekly_Sales'] = submission['Weekly_Sales'].fillna(0)

submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv written')
submission.head(10)

In [ ]:
# Submit to Kaggle directly from Colab
! kaggle competitions submit -c Walmart-Recruiting-Store-Sales-Forecasting -f submission.csv -m "Light_GBM_Training_Best (from MLflow Model Registry)"

In [ ]:
# Check the submission score
! kaggle competitions submissions -c Walmart-Recruiting-Store-Sales-Forecasting | head -5